In [1]:
# PCA Analysis & Visualization 

# This notebook visualizes PCA results including:
# 1. Combined PCA plot of all samples.
# 2. Combined PCA across all chromosomes (color-coded by chromosome).
# 3. PCA per chromosome.
# 4. Genome-wide missingness plot.



In [36]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
import re

In [24]:
# read necessary files

# Paths
input_dir = Path("analysis_results/pca/files/")
output_dir = Path("analysis_results/pca/plots")
output_dir.mkdir(parents=True, exist_ok=True)

# Load FAM file for sample info
fam_df = pd.read_csv(input_dir / "haploid_variants.fam", sep=r"\s+", header=None, usecols=[0, 1], names=["FID", "IID"])



In [ ]:
# 1. Plot Combined PCA plot of all samples.

pca_df = pd.read_csv(input_dir / "pca_results.eigenvec", sep=r"\s+", header=None) 
eigenvalues = pd.read_csv(input_dir / "pca_results.eigenval", sep=r"\s+", header=None) 

# Assign column names to PCA DataFrame
pca_df.columns = ["FID", "IID", "PC1", "PC2"] + [f"PC{i}" for i in range(3, pca_df.shape[1] - 1)]
pca_df["Sample_Label"] = pca_df["IID"]

# Plot PCA
plt.figure(figsize=(8, 6))

# Improved scatterplot with colors and markers
sns.scatterplot(
    x="PC1", y="PC2", data=pca_df, hue="Sample_Label", style="Sample_Label",
    palette="tab10", s=100, edgecolor="black"
)

# Add text labels slightly above points
for i, row in pca_df.iterrows():
    plt.text(row["PC1"], row["PC2"] + 0.02, row["Sample_Label"], 
             fontsize=9, ha='center', weight='bold')

variance_explained = eigenvalues[0] / eigenvalues[0].sum() * 100
plt.xlabel(f"Principal Component 1 ({variance_explained[0]:.2f}%)")
plt.ylabel(f"Principal Component 2 ({variance_explained[1]:.2f}%)")
plt.title("PCA")
plt.legend(title="Samples", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(output_dir / "pca_combined_samples.png", dpi=300)



In [ ]:
# 2. Plot Combined PCA across all chromosomes (color-coded by samples).

# # Load PCA result files per chromosome
pca_files = glob.glob(str(input_dir / "pca_*.eigenvec"))
pca_list = []

# Process each PCA file
for file in pca_files:
    df = pd.read_csv(file, sep=r"\s+", header=None)
    num_pcs = df.shape[1] - 2
    df.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, num_pcs + 1)]
    df = df.merge(fam_df, on=["FID", "IID"], how="left")
    df["Sample"] = df["FID"]
    # Extract based on your chr
    df["Chromosome"] = "_".join(Path(file).stem.split("_")[1:]) 
    pca_list.append(df)

# Merge into a single DataFrame
pca_df = pd.concat(pca_list, ignore_index=True)

## 📊 Plot 1: Combined PCA of All Samples (Merged Chromosomes)

plt.figure(figsize=(10, 8))
sns.scatterplot(x="PC1", y="PC2", data=pca_df, hue="Sample", style="Sample", palette="tab10", s=100, alpha=0.7, edgecolor="k")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Combined PCA: All Samples (Merged Chromosomes)")
plt.legend(title="Sample", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig(output_dir / "pca_combined_samples.png", dpi=300)
plt.show()


In [ ]:
## Plot 3: PCA per Chromosome

# Function to extract a sortable number from the chromosome name
def extract_chrom_num(chrom):
    match = re.search(r'(\d+)', chrom)
    return int(match.group(1)) if match else float('inf')
    
for chrom in sorted(pca_df["Chromosome"].unique(), key=extract_chrom_num):
    df_chr = pca_df[pca_df["Chromosome"] == chrom]
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x="PC1", y="PC2", data=df_chr, hue="Sample", style="Sample", palette="tab10", s=100, edgecolor="black")
    for _, row in df_chr.iterrows():
        plt.text(row["PC1"], row["PC2"] + 0.01, row["Sample"], fontsize=9, ha='center')
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"PCA: Chromosome {chrom}")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_dir / f"pca_chr_{chrom}.png", dpi=300)
    plt.show()


In [ ]:
## Plot 4: Missingness per Sample
imiss_path = input_dir / "plink.imiss"
if imiss_path.exists():
    imiss_df = pd.read_csv(imiss_path, sep=r"\s+")
    imiss_df["F_MISS"] = imiss_df["F_MISS"] * 100
    imiss_df = imiss_df.sort_values("F_MISS", ascending=False)
    # Convert fraction to percentage

    plt.figure(figsize=(10, 6))
    sns.barplot(x="FID", y="F_MISS", data=imiss_df, palette="Blues_r")
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Sample")
    plt.ylabel("Fraction Missing (%)")
    plt.title("Genome-Wide Missingness per Sample")
    plt.tight_layout()
    plt.savefig(output_dir / "missingness.png", dpi=300)
    plt.show()